# 03b · Delta tables: persistence and validation

**Purpose.** Show that every table in the pipeline was really written as Delta and can be read back, and add the one reference table the models need.

1. List where each table lives.
2. Load the Phase 2 split assignment into a Delta table.
3. Read every table back from the catalog and verify its format, row count, columns and key.
4. Report schemas, the row-count flow from Bronze to Gold, and the Delta transaction history.

**Inputs**

- Delta tables written by `01_bronze_layer`, `02_silver_layer` and `03_gold_layer`
- `/Volumes/workspace/car_insurance/raw/split_assignment.csv` (from `src/make_split.py`)
- `/Volumes/workspace/car_insurance/raw/Car_Insurance_Claim.csv` (only to count the source records)

**Outputs.** `workspace.car_insurance.claims_model_split` (Delta): one row per driver `ID` with its Phase 2 split.

**Assumptions**

- Notebooks 01, 02 and 03 have been run. If a table is missing, the validation table says which notebook to run.
- Serverless compute. The notebook can be rerun from top to bottom.

In [0]:
%run ./00_setup

## 1. Table locations

All tables are managed Delta tables in Unity Catalog. The full name (`catalog.schema.table`) is the location to use in any notebook or SQL query.

In [0]:
display(spark.createDataFrame(
    [(c["layer"], name, c["written_by"], c["description"]) for name, c in TABLE_CONTRACT.items()],
    "layer string, table string, written_by string, contents string"))

## 2. Split reference table

Phase 2 split the drivers with scikit-learn (`train_test_split`, `random_state=123`): first 80/20 into training and test, then 80/20 of the training drivers into train and validation, stratified on the outcome. Spark's `randomSplit` cannot reproduce that shuffle. `src/make_split.py` therefore reruns the two scikit-learn calls and saves which split each driver `ID` falls in.

Storing that lookup as a Delta table lets every Spark model train and test on exactly the same drivers as Phase 2, which is what makes the comparison fair. The lookup holds only `ID` and the split name: no features and no outcome.

In [0]:
require_source_files(SPLIT_CSV, CLAIMS_CSV)

split_file_records = spark.read.text(SPLIT_CSV).count() - 1   # minus the header line

split_df = (
    spark.read.format("csv")
    .schema("ID string, split string")
    .option("header", "true")
    .load(SPLIT_CSV)
    .select(F.expr("try_cast(ID AS bigint)").alias("ID"), F.lower(F.trim("split")).alias("split"))
)

write_delta(split_df, SPLIT_TABLE)

In [0]:
split_table = spark.table(SPLIT_TABLE)   # read back from the catalog

unexpected = split_table.filter(F.col("ID").isNull() | ~F.col("split").isin(*SPLIT_NAMES)).count()
assert unexpected == 0, f"{unexpected} rows have a missing ID or an unknown split name"

display(split_table.groupBy("split").count().orderBy("split"))

## 3. Read back and verify every table

Each table is read from the catalog, not from a DataFrame still in memory, and checked for:

- **format**: stored as Delta
- **rows**: not empty, and equal to the expected count where one is known. Bronze must hold exactly one row per record in the source file, counted here directly from the raw text.
- **columns**: the columns the next notebook relies on are present
- **key**: `ID` is unique and never missing in the row-level tables

The notebook stops here if any table fails.

In [0]:
source_records = spark.read.text(CLAIMS_CSV).count() - 1   # minus the header line
print(f"Records in the source file: {source_records:,}")

validate_tables({
    BRONZE_TABLE: source_records,
    SILVER_TABLE: None,          # cleaning may remove rows; the flow is reported below
    GOLD_FEATURES_TABLE: None,
    GOLD_SUMMARY_TABLE: None,
    SPLIT_TABLE: split_file_records,
})

## 4. Schemas as saved

In [0]:
display(spark.createDataFrame(
    [(name, position + 1, column, data_type)
     for name in TABLE_CONTRACT
     for position, (column, data_type) in enumerate(spark.table(name).dtypes)],
    "table string, position int, column string, data_type string"))

## 5. Row counts from Bronze to Gold

How many drivers each layer keeps, and how many were removed on the way.

In [0]:
row_counts = {name: spark.table(name).count()
              for name in (BRONZE_TABLE, SILVER_TABLE, GOLD_FEATURES_TABLE)}

flow, previous = [("Source file", source_records, None)], source_records
for name, rows in row_counts.items():
    flow.append((name, rows, previous - rows))
    previous = rows

display(spark.createDataFrame(flow, "stage string, rows long, rows_removed_from_previous long"))

### Split coverage of the Gold feature table

Phase 2 used 6,400 train, 1,600 validation and 2,000 test drivers, with 656 claims in the test set. If cleaning removed drivers, the counts below fall short of those numbers, and the model comparison with Phase 2 is no longer on identical test drivers. That would not be an error, but it has to be stated in the report.

In [0]:
PHASE2_SPLIT_SIZES = {"train": 6400, "validation": 1600, "test": 2000}

coverage = {name: split.agg(F.count("*").alias("drivers"),
                            F.sum(F.col(LABEL_COLUMN).cast("int")).alias("claims")).first()
            for name, split in zip(SPLIT_NAMES, load_model_splits())}

display(spark.createDataFrame(
    [(name, PHASE2_SPLIT_SIZES[name], int(row["drivers"]), int(row["claims"] or 0),
      "same drivers as Phase 2" if row["drivers"] == PHASE2_SPLIT_SIZES[name]
      else f"{PHASE2_SPLIT_SIZES[name] - row['drivers']:,} Phase 2 drivers missing")
     for name, row in coverage.items()],
    "split string, phase2_drivers long, gold_drivers long, gold_claims long, note string"))

unassigned = row_counts[GOLD_FEATURES_TABLE] - sum(int(row["drivers"]) for row in coverage.values())
assert unassigned == 0, f"{unassigned} Gold drivers have no split assignment"

## 6. Delta transaction history

Delta records every write as a new table version. The table below shows, for each table, its latest version and when its data was last written; rerunning a notebook adds a version instead of silently replacing files.

In [0]:
history = []
for name in TABLE_CONTRACT:
    versions = spark.sql(f"DESCRIBE HISTORY {name}").select("version", "timestamp", "operation")
    latest_version = versions.agg(F.max("version")).first()[0]
    # Data writes are logged as WRITE or CREATE OR REPLACE TABLE AS SELECT; comments are not.
    last_write = (versions
                  .filter((F.col("operation") == "WRITE") | F.col("operation").contains("AS SELECT"))
                  .orderBy(F.desc("version")).first())
    if last_write is None:   # operation name not recognised: fall back to the latest version
        last_write = versions.orderBy(F.desc("version")).first()
    history.append((name, int(latest_version), int(last_write["version"]),
                    str(last_write["timestamp"]), last_write["operation"]))

display(spark.createDataFrame(
    history,
    "table string, latest_version long, last_data_write_version long, last_data_write string, operation string"))

In [0]:
# Time travel: an earlier version of a table can still be queried.
try:
    first_version_rows = spark.sql(f"SELECT COUNT(*) AS n FROM {SPLIT_TABLE} VERSION AS OF 0").first()["n"]
    print(f"{SPLIT_TABLE}: version 0 has {first_version_rows:,} rows, "
          f"the current version has {spark.table(SPLIT_TABLE).count():,} rows.")
except Exception as error:   # old versions are removed once their retention period has passed
    print("Version 0 is no longer available for time travel:", str(error)[:200])

**Result.** Every pipeline table exists as a Delta table in `workspace.car_insurance`, was read back from the catalog, and passed the format, row-count, column and key checks in section 3. The split reference table gives the model notebooks the same train, validation and test drivers as Phase 2.

Next: the model notebooks (`04a`, `04b`, `04c`).